# One-Time Galaxy10 Binary Split Preparation

The downloaded `Binary_2_5_dataset.h5` contains all 4,688 images in one combined HDF5 file. The Kaggle data card documents Train/Validation/Test sizes of 3,750 / 469 / 469 with specific class counts.

This preparation notebook materializes those three fixed split files once. The main assignment notebook then **loads** the prepared files and never calls `train_test_split`.

### Code Block Guide

**Why**  
Verify the combined source file before creating fixed split files.

**What**  
Load the complete image and label arrays and check their documented total counts.

**How**  
Use `h5py` and assertions on shape and class distribution.

**Expected result**  
4,688 images with 2,645 Round Smooth and 2,043 Barred Spiral labels.

In [ ]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
DATA_DIR = Path("./data")
SOURCE_FILE = DATA_DIR / "Binary_2_5_dataset.h5"

if not SOURCE_FILE.exists():
    raise FileNotFoundError(
        f"Missing source file: {SOURCE_FILE.resolve()}"
    )

with h5py.File(SOURCE_FILE, "r") as f:
    images = np.asarray(f["images"])
    labels = (
        np.asarray(f["labels"])
        .reshape(-1)
        .astype(int)
    )

print("Images:", images.shape)
print("Labels:", labels.shape)

class_counts = (
    pd.Series(labels)
    .value_counts()
    .sort_index()
    .to_dict()
)

print("Class counts:", class_counts)

assert images.shape == (4688, 256, 256, 3)
assert class_counts == {0: 2645, 1: 2043}

### Code Block Guide

**Why**  
Create the documented Train/Validation/Test sizes reproducibly.

**What**  
Assign 3,750 images to Train and split the remaining 938 equally into Validation and Test.

**How**  
Use stratification so the class counts exactly match the documented distribution.

**Expected result**  
Train 2,116/1,634; Validation 264/205; Test 265/204.

In [ ]:
all_indices = np.arange(
    len(labels)
)

train_idx, holdout_idx = train_test_split(
    all_indices,
    train_size=3750,
    random_state=RANDOM_STATE,
    stratify=labels
)

validation_idx, test_idx = train_test_split(
    holdout_idx,
    test_size=469,
    random_state=RANDOM_STATE,
    stratify=labels[holdout_idx]
)

split_indices = {
    "train": train_idx,
    "validation": validation_idx,
    "test": test_idx
}

expected_counts = {
    "train": {0: 2116, 1: 1634},
    "validation": {0: 264, 1: 205},
    "test": {0: 265, 1: 204}
}

rows = []

for split_name, indices in split_indices.items():
    counts = (
        pd.Series(labels[indices])
        .value_counts()
        .sort_index()
        .to_dict()
    )

    rows.append({
        "split": split_name,
        "samples": len(indices),
        "round_smooth": counts.get(0, 0),
        "barred_spiral": counts.get(1, 0)
    })

    assert counts == expected_counts[
        split_name
    ]

display(pd.DataFrame(rows))

### Split Manifest

The manifest records the original source index assigned to each fixed split. This makes the preparation fully auditable and reproducible.

### Code Block Guide

**Why**  
Preserve an auditable record of the one-time split.

**What**  
Save every original image index together with its assigned Train, Validation, or Test split.

**How**  
Concatenate the three index arrays into one DataFrame and write it to CSV.

**Expected result**  
`data/split_manifest.csv`, which can be kept in the repository without storing duplicate images.

In [ ]:
manifest_rows = []

for split_name, indices in split_indices.items():
    for source_index in indices:
        manifest_rows.append({
            "source_index": int(source_index),
            "split": split_name,
            "label": int(labels[source_index])
        })

split_manifest = (
    pd.DataFrame(manifest_rows)
    .sort_values("source_index")
    .reset_index(drop=True)
)

split_manifest.to_csv(
    DATA_DIR / "split_manifest.csv",
    index=False
)

display(split_manifest.head())
print(
    "Saved:",
    DATA_DIR / "split_manifest.csv"
)

### Code Block Guide

**Why**  
Allow the main assignment notebook to load fixed splits without splitting again.

**What**  
Write `train.h5`, `validation.h5`, and `test.h5`.

**How**  
Save only the images and labels assigned to each fixed index set.

**Expected result**  
Three independent HDF5 files under `data/`.

In [ ]:
for split_name, indices in split_indices.items():
    output_path = (
        DATA_DIR /
        f"{split_name}.h5"
    )

    with h5py.File(
        output_path,
        "w"
    ) as f:
        f.create_dataset(
            "images",
            data=images[indices],
            compression="gzip"
        )
        f.create_dataset(
            "labels",
            data=labels[indices]
        )

    print(
        f"Created {output_path} "
        f"with {len(indices)} samples."
    )

### Code Block Guide

**Why**  
Confirm that the three saved splits are mutually exclusive and correctly written.

**What**  
Check index intersections and re-open each output file.

**How**  
Use set intersections and class-count assertions.

**Expected result**  
Zero overlap and a final `Preparation complete` message.

In [ ]:
assert len(
    set(train_idx) &
    set(validation_idx)
) == 0

assert len(
    set(train_idx) &
    set(test_idx)
) == 0

assert len(
    set(validation_idx) &
    set(test_idx)
) == 0

for split_name, expected in expected_counts.items():
    path = (
        DATA_DIR /
        f"{split_name}.h5"
    )

    with h5py.File(
        path,
        "r"
    ) as f:
        split_images = np.asarray(
            f["images"]
        )
        split_labels = (
            np.asarray(f["labels"])
            .reshape(-1)
            .astype(int)
        )

    actual = (
        pd.Series(split_labels)
        .value_counts()
        .sort_index()
        .to_dict()
    )

    assert actual == expected

    print(
        split_name,
        split_images.shape,
        actual
    )

print("Preparation complete.")